# 第7章 Notebook：2人恋愛モデル

対応章: [`../chapters/07_love_dynamics_basic.md`](../chapters/07_love_dynamics_basic.md)

この notebook は、卒業研究準備セミナーの数値実験用である。上から順に実行すれば、本文で説明した図を再現できる。設定パラメータは上部のセルにまとめてある。乱数は seed を固定している。

## 1. ライブラリ読み込み

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

np.random.seed(0)

## 2. パラメータ設定：係数行列 M

$\dot{R}=aR+bJ,\ \dot{J}=cR+dJ$。

In [ ]:
a, b = 0.0, 1.0
c, d = -1.0, 0.0
M = np.array([[a, b], [c, d]])
eig = np.linalg.eigvals(M)
print('eigenvalues:', eig)

## 3-4. ODE の定義と数値積分

In [ ]:
def rhs(t, y, M, force=None):
    dy = M @ y
    if force is not None:
        dy = dy + force(t)
    return dy

t_span = (0, 20)
t_eval = np.linspace(*t_span, 400)
y0 = [1.0, 0.0]
sol = solve_ivp(rhs, t_span, y0, args=(M,), t_eval=t_eval)

## 5. 時系列プロット

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(sol.t, sol.y[0], label='R (Romeo)')
plt.plot(sol.t, sol.y[1], label='J (Juliet)')
plt.xlabel('time'); plt.ylabel('affection'); plt.legend()
plt.title('two-person love dynamics'); plt.tight_layout(); plt.show()

## 6. 相図（ベクトル場 + 軌道）

In [ ]:
def phase_portrait(M, ax, span=2.0, n=15):
    r = np.linspace(-span, span, n)
    R, J = np.meshgrid(r, r)
    dR = M[0, 0] * R + M[0, 1] * J
    dJ = M[1, 0] * R + M[1, 1] * J
    ax.quiver(R, J, dR, dJ, color='gray', alpha=0.6)

fig, ax = plt.subplots(figsize=(5, 5))
phase_portrait(M, ax)
ax.plot(sol.y[0], sol.y[1], 'b', lw=2)
ax.plot(y0[0], y0[1], 'ko')
ax.set_xlabel('R'); ax.set_ylabel('J'); ax.set_title('phase portrait')
plt.tight_layout(); plt.show()

## 7. 外力イベントの追加

ある時刻にロミオへ正のパルス（うれしい出来事）を加える。安定焦点型の係数で比較する。

In [ ]:
Ms = np.array([[-0.2, 1.0], [-1.0, -0.2]])  # stable focus
def force(t):
    return np.array([2.0 * np.exp(-((t - 5)**2) / 0.5), 0.0])

sol_free = solve_ivp(rhs, t_span, y0, args=(Ms,), t_eval=t_eval)
sol_forced = solve_ivp(rhs, t_span, y0, args=(Ms, force), t_eval=t_eval)
plt.figure(figsize=(6, 4))
plt.plot(sol_free.t, sol_free.y[0], '--', label='R no event')
plt.plot(sol_forced.t, sol_forced.y[0], label='R with event')
plt.xlabel('time'); plt.ylabel('R affection'); plt.legend()
plt.title('effect of an external event'); plt.tight_layout(); plt.show()

## 8. 課題（自分で変更する）

1. 鞍点型（`det M < 0`）の係数を選び、時系列と相図がどうなるか確かめよ。
2. 外力パルスの時刻を変えると、最終的な関係が変わるか調べよ。

In [ ]:
# === 課題セル ===
for label_, Mtry in [('vortex', np.array([[0,1.],[-1,0]])),
                     ('saddle', np.array([[1.,0.5],[0.5,-1.]])),
                     ('stable focus', np.array([[-0.3,1.],[-1.,-0.3]]))]:
    print(f'{label_:14s} eigenvalues = {np.linalg.eigvals(Mtry)}')